# The Annotated Transformer — versão ilustrada para Google Colab

Esta é uma adaptação didática inspirada em **The Annotated Transformer**, do Harvard NLP.
O objetivo aqui é manter a intuição central do artigo, mas em um notebook mais estável para **Google Colab**,
com **plots**, **heatmaps**, **diagramas** e um **exemplo treinável** em pequena escala.

## O que este notebook cobre

1. Visão geral da arquitetura Encoder-Decoder
2. Embeddings e Positional Encoding
3. Máscara causal no decoder
4. Scaled Dot-Product Attention
5. Multi-Head Attention
6. Implementação mínima do Transformer em PyTorch
7. Treino em uma tarefa sintética de cópia
8. Visualização de loss, learning rate e mapas de atenção

> Observação: esta versão prioriza didática e reprodutibilidade no Colab.

## Setup para Google Colab

A célula abaixo instala apenas o necessário para o notebook rodar tanto em CPU quanto em GPU.

In [ ]:

# Se estiver no Google Colab, esta instalação é suficiente.
# Em ambientes locais, pode comentar esta célula se já tiver as dependências.
!pip -q install torch matplotlib pandas numpy

In [ ]:

import copy
import math
import random
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.nn.functional import log_softmax
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

## 1. Visão geral da arquitetura

O Transformer clássico possui:

- **Encoder stack**: lê a sequência de entrada
- **Decoder stack**: gera a sequência de saída
- **Self-attention**: cada token pode "olhar" para outros tokens
- **Cross-attention**: o decoder consulta a representação produzida pelo encoder
- **Feed-forward network**: transformação não linear aplicada por posição

In [ ]:

from matplotlib.patches import Rectangle, FancyArrowPatch

def draw_transformer_overview():
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 8)
    ax.axis("off")

    def box(x, y, w, h, label, fs=10):
        rect = Rectangle((x, y), w, h, fill=False, linewidth=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=fs)

    # Encoder stack
    for i in range(3):
        box(1.0, 1.0 + i*1.6, 2.5, 1.1, f"Encoder Layer {i+1}")
    ax.text(2.25, 6.3, "Encoder Stack", ha="center", fontsize=12, fontweight="bold")

    # Decoder stack
    for i in range(3):
        box(8.5, 1.0 + i*1.6, 2.5, 1.1, f"Decoder Layer {i+1}")
    ax.text(9.75, 6.3, "Decoder Stack", ha="center", fontsize=12, fontweight="bold")

    # Inputs / outputs
    box(0.3, 0.2, 1.4, 0.5, "Input")
    box(7.8, 0.2, 1.4, 0.5, "Shifted Output")
    box(10.0, 0.2, 1.6, 0.5, "Next Token")

    # Embeddings
    box(0.3, 0.9, 1.4, 0.6, "Emb + PE")
    box(7.8, 0.9, 1.4, 0.6, "Emb + PE")

    def arrow(x1, y1, x2, y2, style="-|>"):
        arr = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle=style, mutation_scale=15, linewidth=1.7)
        ax.add_patch(arr)

    # Vertical arrows
    arrow(1.0, 0.7, 1.0, 0.9)
    arrow(8.5, 0.7, 8.5, 0.9)
    arrow(2.0, 1.5, 2.0, 1.9)
    arrow(9.3, 1.5, 9.3, 1.9)
    arrow(9.75, 5.9, 10.8, 0.7)

    # Connections
    arrow(1.7, 1.2, 1.0, 1.55)
    arrow(2.25, 5.45, 8.5, 4.9)
    ax.text(5.4, 5.35, "Encoder outputs (memory)", ha="center", fontsize=10)

    plt.title("Arquitetura geral do Transformer", fontsize=14)
    plt.show()

draw_transformer_overview()

## 2. Positional Encoding

Como o Transformer não possui recorrência, ele precisa de uma forma explícita de incorporar a ordem dos tokens.

No artigo original, isso é feito com seno e cosseno:

\[
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right),
\qquad
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
\]

Abaixo, implementamos isso e plotamos algumas dimensões.

In [ ]:

def positional_encoding(max_len: int, d_model: int):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe = positional_encoding(max_len=100, d_model=32)

plt.figure(figsize=(12, 4))
for i in [0, 1, 2, 3, 8, 9]:
    plt.plot(pe[:, i].numpy(), label=f"dim {i}")
plt.title("Positional Encoding em algumas dimensões")
plt.xlabel("Posição")
plt.ylabel("Valor")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:

plt.figure(figsize=(10, 6))
plt.imshow(pe[:, :32].T.numpy(), aspect="auto")
plt.colorbar()
plt.title("Heatmap do Positional Encoding")
plt.xlabel("Posição")
plt.ylabel("Dimensão")
plt.show()

## 3. Máscara causal no decoder

Durante treinamento autoregressivo, o token na posição `t` não pode enxergar o futuro.
Essa restrição é implementada com uma **subsequent mask** triangular.

In [ ]:

def subsequent_mask(size: int):
    attn_shape = (1, size, size)
    mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(torch.uint8)
    return mask == 0

mask = subsequent_mask(12)[0].int()

plt.figure(figsize=(6, 5))
plt.imshow(mask.numpy(), aspect="auto")
plt.title("Máscara causal (1 = permitido, 0 = bloqueado)")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.colorbar()
plt.show()

mask

## 4. Scaled Dot-Product Attention

A operação central é:

\[
Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

Abaixo fazemos um exemplo pequeno e visualizamos a matriz de atenção.

In [ ]:

def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

set_seed(42)
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

attended, attn_map = attention(Q, K, V)
print("Output shape:", attended.shape)
print("Attention map shape:", attn_map.shape)

plt.figure(figsize=(6, 5))
plt.imshow(attn_map[0].detach().numpy(), aspect="auto")
plt.colorbar()
plt.title("Mapa de atenção (toy example)")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.show()

## 5. Multi-Head Attention

Em vez de uma única projeção, o Transformer usa várias cabeças em paralelo.
Isso permite que o modelo capture diferentes padrões relacionais ao mesmo tempo.

In [ ]:

def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super().__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            # broadcast over heads
            mask = mask.unsqueeze(1)

        nbatches = query.size(0)

        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears[:3], (query, key, value))
        ]

        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)

        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        return self.linears[-1](x)

mha = MultiHeadedAttention(h=4, d_model=32)
x = torch.randn(2, 10, 32)
y = mha(x, x, x)
print("Input:", x.shape)
print("Output:", y.shape)
print("Attention tensor:", mha.attn.shape)

In [ ]:

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for head in range(4):
    axes[head].imshow(mha.attn[0, head].detach().numpy(), aspect="auto")
    axes[head].set_title(f"Head {head}")
    axes[head].set_xlabel("Key")
    axes[head].set_ylabel("Query")
plt.suptitle("Mapas de atenção por cabeça")
plt.tight_layout()
plt.show()

## 6. Implementação mínima do Transformer

Agora montamos uma versão compacta e didática das peças principais do artigo.

In [ ]:

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(torch.relu(self.w_1(x))))

class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = positional_encoding(max_len=max_len, d_model=d_model)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)
        self.size = size

    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)

class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)

class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

    def forward(self, src, tgt, src_mask, tgt_mask):
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

def make_model(src_vocab, tgt_vocab, N=2, d_model=64, d_ff=128, h=4, dropout=0.1):
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model, dropout)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)

    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab),
    )

    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    return model

## 7. Dados sintéticos: copy task

Para manter o notebook leve e executável no Colab, usamos uma tarefa sintética simples:
o modelo recebe uma sequência e deve reproduzi-la na saída.
Isso é suficiente para mostrar o fluxo de treinamento e inferência.

In [ ]:

@dataclass
class Batch:
    src: torch.Tensor
    tgt: torch.Tensor
    src_mask: torch.Tensor
    tgt_mask: torch.Tensor
    tgt_y: torch.Tensor
    ntokens: int

def make_std_mask(tgt, pad):
    tgt_mask = (tgt != pad).unsqueeze(-2)
    tgt_mask = tgt_mask & subsequent_mask(tgt.size(-1)).type_as(tgt_mask.data)
    return tgt_mask

def generate_copy_data(batch_size, seq_len, vocab_size, pad_idx=0, bos_idx=1):
    src = torch.randint(2, vocab_size, (batch_size, seq_len), device=device)
    src[:, 0] = bos_idx
    tgt = src.clone()
    return src, tgt

def make_batch(src, tgt, pad_idx=0):
    src_mask = (src != pad_idx).unsqueeze(-2)
    tgt_input = tgt[:, :-1]
    tgt_y = tgt[:, 1:]
    tgt_mask = make_std_mask(tgt_input, pad_idx)
    ntokens = (tgt_y != pad_idx).data.sum().item()
    return Batch(
        src=src,
        tgt=tgt_input,
        src_mask=src_mask,
        tgt_mask=tgt_mask,
        tgt_y=tgt_y,
        ntokens=ntokens,
    )

In [ ]:

class LabelSmoothing(nn.Module):
    def __init__(self, size, padding_idx, smoothing=0.0):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction="sum")
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size

    def forward(self, x, target):
        true_dist = x.data.clone()
        true_dist.fill_(self.smoothing / (self.size - 2))
        true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = torch.nonzero(target.data == self.padding_idx, as_tuple=False)
        if mask.numel() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        return self.criterion(x, true_dist.detach())

class SimpleLossCompute:
    def __init__(self, generator, criterion):
        self.generator = generator
        self.criterion = criterion

    def __call__(self, x, y, norm):
        x = self.generator(x)
        sloss = self.criterion(x.contiguous().view(-1, x.size(-1)), y.contiguous().view(-1)) / norm
        return sloss

class NoamOpt:
    def __init__(self, model_size, factor, warmup, optimizer):
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.model_size = model_size
        self._rate = 0

    def step(self):
        self._step += 1
        rate = self.rate()
        for p in self.optimizer.param_groups:
            p["lr"] = rate
        self._rate = rate
        self.optimizer.step()

    def zero_grad(self):
        self.optimizer.zero_grad()

    def rate(self, step=None):
        if step is None:
            step = self._step
        return self.factor * (
            self.model_size ** (-0.5) *
            min(step ** (-0.5), step * self.warmup ** (-1.5))
        )

In [ ]:

def run_epoch(model, optimizer, criterion, vocab_size, num_batches=20, batch_size=48, seq_len=12, train=True):
    model.train(train)
    total_loss = 0.0
    total_tokens = 0
    lrs = []

    for _ in range(num_batches):
        src, tgt = generate_copy_data(batch_size=batch_size, seq_len=seq_len, vocab_size=vocab_size)
        batch = make_batch(src, tgt)

        out = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
        loss = criterion(out, batch.tgt_y, batch.ntokens)

        if train:
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            lrs.append(optimizer._rate)

        total_loss += loss.item() * batch.ntokens
        total_tokens += batch.ntokens

    return total_loss / total_tokens, lrs

In [ ]:

VOCAB_SIZE = 32
PAD_IDX = 0
BOS_IDX = 1

model = make_model(
    src_vocab=VOCAB_SIZE,
    tgt_vocab=VOCAB_SIZE,
    N=2,
    d_model=64,
    d_ff=128,
    h=4,
    dropout=0.1,
).to(device)

criterion = SimpleLossCompute(
    model.generator,
    LabelSmoothing(size=VOCAB_SIZE, padding_idx=PAD_IDX, smoothing=0.0),
)

optimizer = NoamOpt(
    model_size=64,
    factor=1.0,
    warmup=200,
    optimizer=Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
)

history = {"train_loss": [], "val_loss": [], "lr": []}
EPOCHS = 8

for epoch in range(EPOCHS):
    train_loss, train_lrs = run_epoch(model, optimizer, criterion, VOCAB_SIZE, num_batches=20, train=True)
    with torch.no_grad():
        val_loss, _ = run_epoch(model, optimizer, criterion, VOCAB_SIZE, num_batches=4, train=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].extend(train_lrs)
    print(f"Epoch {epoch+1:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

## 8. Curvas de treinamento

Vamos visualizar a evolução da loss e da learning rate usada no scheduler do artigo.

In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="validation")
plt.title("Loss por época")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(history["lr"])
plt.title("Learning rate schedule (Noam)")
plt.xlabel("Passo de otimização")
plt.ylabel("Learning rate")
plt.grid(alpha=0.3)
plt.show()

## 9. Decoding guloso

Agora testamos o modelo em inferência.

In [ ]:

def greedy_decode(model, src, src_mask, max_len, start_symbol):
    memory = model.encode(src, src_mask)
    ys = torch.ones(1, 1, dtype=torch.long, device=src.device).fill_(start_symbol)
    for _ in range(max_len - 1):
        out = model.decode(memory, src_mask, ys, make_std_mask(ys, pad=0))
        prob = model.generator(out[:, -1])
        next_word = torch.argmax(prob, dim=1).item()
        next_word_tensor = torch.tensor([[next_word]], dtype=torch.long, device=src.device)
        ys = torch.cat([ys, next_word_tensor], dim=1)
    return ys

model.eval()
src = torch.tensor([[1, 7, 5, 6, 9, 4, 3, 2, 8, 11, 10, 12]], device=device)
src_mask = (src != 0).unsqueeze(-2)
decoded = greedy_decode(model, src, src_mask, max_len=src.size(1), start_symbol=1)

print("Entrada :", src[0].tolist())
print("Saída   :", decoded[0].tolist())

## 10. Visualizando a atenção aprendida

Vamos inspecionar:

- a atenção do **encoder**
- a atenção mascarada do **decoder**
- a **cross-attention** entre decoder e encoder

In [ ]:

# Rodamos uma passada completa para popular os tensores de atenção
with torch.no_grad():
    batch = make_batch(src, src)
    _ = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)

encoder_attn = model.encoder.layers[0].self_attn.attn[0].detach().cpu()
decoder_self_attn = model.decoder.layers[0].self_attn.attn[0].detach().cpu()
decoder_src_attn = model.decoder.layers[0].src_attn.attn[0].detach().cpu()

print("encoder_attn:", encoder_attn.shape)
print("decoder_self_attn:", decoder_self_attn.shape)
print("decoder_src_attn:", decoder_src_attn.shape)

In [ ]:

def plot_heads(attn_tensor, title_prefix):
    n_heads = attn_tensor.size(0)
    fig, axes = plt.subplots(1, n_heads, figsize=(4*n_heads, 4))
    if n_heads == 1:
        axes = [axes]
    for h in range(n_heads):
        axes[h].imshow(attn_tensor[h].numpy(), aspect="auto")
        axes[h].set_title(f"{title_prefix} - head {h}")
        axes[h].set_xlabel("Key")
        axes[h].set_ylabel("Query")
    plt.tight_layout()
    plt.show()

plot_heads(encoder_attn, "Encoder self-attention")
plot_heads(decoder_self_attn, "Decoder masked self-attention")
plot_heads(decoder_src_attn, "Decoder cross-attention")

## 11. Interpretação rápida

O que observar:

- **Encoder self-attention**: relacionamentos internos na sequência de entrada
- **Decoder masked self-attention**: padrão triangular devido à máscara causal
- **Cross-attention**: quais posições de saída consultam quais posições da entrada

Esses três blocos formam o núcleo da arquitetura original descrita no artigo.

## 12. Próximos passos

Se você quiser estender este notebook, os próximos passos naturais são:

1. trocar a copy task por um dataset real
2. adicionar tokenização e vocabulário
3. treinar em tradução ou language modeling
4. comparar com `torch.nn.Transformer`
5. instrumentar métricas e logging mais detalhados

---

Notebook preparado para execução em **Google Colab** com foco didático.